In [9]:
import os
#os.chdir("../")
%pwd

'/home/abood/datascienceproject'

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("artifacts/data_ingestion/machine.data", header=None, names=["vendor","model","MYCT","MMIN","MMAX","CACH","CHMIN","CHMAX","PRP","ERP"])
df = pd.DataFrame(data)
df.head()

,vendor,model,MYCT,MMIN,MMAX,CACH,CHMIN,CHMAX,PRP,ERP
0,adviser,32/60,125,256,6000,256,16,128,198,199
1,amdahl,470v/7,29,8000,32000,32,8,32,269,253
2,amdahl,470v/7a,29,8000,32000,32,8,32,220,253
3,amdahl,470v/7b,29,8000,32000,32,8,32,172,253
4,amdahl,470v/7c,29,8000,16000,32,8,16,132,132


In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [12]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path
        )
        return data_transformation_config

In [14]:
import os
from src.datascience.utils import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [15]:
class DataTransformation:
    def __init__(self,config: DataTransformationConfig):
        self.config = config
    def train_test_spliting(self):
        data = pd.read_csv(self.config.data_path, header=None, names=["vendor","model","MYCT","MMIN","MMAX","CACH","CHMIN","CHMAX","PRP","ERP"])
        # Drop non-numeric and non-feature columns; keep only schema features + target
        data = data.drop(columns=["vendor", "model", "ERP"])
        train , test = train_test_split(data)
        train.to_csv(os.path.join(self.config.root_dir,"train.csv"),index=False)
        test.to_csv(os.path.join(self.config.root_dir,"test.csv"),index=False)

        logger.info(f"splitting is done and train and test files are saved in {self.config.root_dir}")
        logger.info(train.shape)
        logger.info(test.shape)

        print(train.shape)
        print(test.shape)

In [16]:
try:
    config = ConfigurationManager()
    data_trainsformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_trainsformation_config)
    data_transformation.train_test_spliting()
except Exception as e:
    logger.exception(e)

[2026-05-06 13:22:30,575: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-06 13:22:30,577: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-06 13:22:30,580: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-06 13:22:30,580: INFO: common]: created directory at: artifacts
[2026-05-06 13:22:30,581: INFO: common]: created directory at: artifacts/data_transformation
[2026-05-06 13:22:30,587: INFO: 32226923]: splitting is done and train and test files are saved in artifacts/data_transformation
[2026-05-06 13:22:30,588: INFO: 32226923]: (156, 7)
[2026-05-06 13:22:30,589: INFO: 32226923]: (53, 7)
(156, 7)
(53, 7)
